# NB01 — Descarga y verificación

Consume los seis archivos oficiales y produce `metadata/`, `data_cruda/` e `inventario_descarga.json`, según `marco_muestral_v2.md` §5.1.

In [ ]:
# CONFIG único — marco v2 §5.1. Sin magics ni pip.
import os, json, time, zipfile
from pathlib import Path, PurePosixPath
import requests
BASE_DIR = Path(os.environ.get('DERM_IAPP_BASE', '/content/derm_iapp'))
CONFIG = {'BASE_DIR': BASE_DIR, 'FULL_DATASET': os.environ.get('DERM_IAPP_FULL_DATASET','1') == '1', 'SEMILLA': 2026, 'VERIFY_CRC': os.environ.get('DERM_IAPP_VERIFY_CRC','0') == '1', 'DELETE_ZIPS_AFTER_EXTRACT': os.environ.get('DERM_IAPP_DELETE_ZIPS_AFTER_EXTRACT','0') == '1', 'TAMANOS_CSV': {'ISIC_2019_Training_Metadata.csv':1214351,'ISIC_2019_Training_GroundTruth.csv':1291479,'ISIC_2019_Test_Metadata.csv':293638,'ISIC_2019_Test_GroundTruth.csv':464810}, 'ZIP_TAMANOS_REFERENCIA': {'ISIC_2019_Training_Input.zip':9771618190,'ISIC_2019_Test_Input.zip':3823247963}}
CONFIG['URLS'] = {n:'https://isic-archive.s3.amazonaws.com/challenges/2019/'+n for n in list(CONFIG['TAMANOS_CSV'])+list(CONFIG['ZIP_TAMANOS_REFERENCIA'])}
_HTTP_GET = requests.get

def _descargar_stream(url, destino, reintentos=3):
    """Descarga atómica por chunks de 1 MiB, con tres intentos."""
    destino=Path(destino); ultimo=None
    for intento in range(1, reintentos+1):
        temporal=destino.with_suffix(destino.suffix+'.part')
        try:
            respuesta=_HTTP_GET(url, stream=True, timeout=60); respuesta.raise_for_status()
            with open(temporal,'wb') as salida:
                for bloque in respuesta.iter_content(1024*1024):
                    if bloque: salida.write(bloque)
            temporal.replace(destino); return destino
        except Exception as exc:
            ultimo=exc; temporal.unlink(missing_ok=True)
            if intento<reintentos: time.sleep(2*intento)
    raise RuntimeError(f'No se pudo descargar {url}: {ultimo}')

def asegurar_archivo(nombre, url, tamano_esperado):
    """R1: CSV byte-exacto; ZIP local es referencia operativa y solo avisa."""
    es_csv=nombre.endswith('.csv'); destino=BASE_DIR/('metadata' if es_csv else '')/nombre; destino.parent.mkdir(parents=True,exist_ok=True)
    if destino.exists():
        observado=destino.stat().st_size
        if es_csv and observado != tamano_esperado: raise RuntimeError(f'tamaño inválido para {nombre}: {observado}, esperado {tamano_esperado}')
        if not es_csv and observado != CONFIG['ZIP_TAMANOS_REFERENCIA'][nombre]: print(f'Aviso: ZIP local {nombre} difiere de referencia')
        return destino, 'ok_existente', observado
    if es_csv:
        cabecera=_HTTP_GET(url,stream=True,timeout=60); cabecera.raise_for_status(); contenido=int(cabecera.headers.get('Content-Length','-1'))
        if contenido != tamano_esperado: raise RuntimeError(f'tamaño Content-Length inválido para {nombre}: {contenido}, esperado {tamano_esperado}')
    _descargar_stream(url,destino); observado=destino.stat().st_size
    if es_csv and observado != tamano_esperado: raise RuntimeError(f'tamaño final inválido para {nombre}: {observado}, esperado {tamano_esperado}')
    return destino, 'descargado', observado

def extraer_zip(origen, particion):
    """Preserva la carpeta interna y descarta miembros con path traversal."""
    destino=BASE_DIR/'data_cruda'/particion; destino.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(origen) as archivo:
        jpg_zip=sum(i.filename.lower().endswith('.jpg') for i in archivo.infolist())
        for info in archivo.infolist():
            ruta=PurePosixPath(info.filename)
            if ruta.is_absolute() or '..' in ruta.parts: continue
            final=destino.joinpath(*ruta.parts); final.parent.mkdir(parents=True,exist_ok=True)
            if not info.is_dir():
                with archivo.open(info) as entrada, open(final,'wb') as salida: salida.write(entrada.read())
        if CONFIG['VERIFY_CRC']: assert archivo.testzip() is None, 'CRC inválido (leer ~12.7 GB tarda)'
    return jpg_zip

def main():
    inventario=[]
    for nombre,tamano in CONFIG['TAMANOS_CSV'].items():
        ruta,estado,observado=asegurar_archivo(nombre,CONFIG['URLS'][nombre],tamano); inventario.append({'nombre':nombre,'url':CONFIG['URLS'][nombre],'tamano_esperado':tamano,'tamano_observado':observado,'estado':estado})
    for nombre,referencia in CONFIG['ZIP_TAMANOS_REFERENCIA'].items():
        ruta,estado,observado=asegurar_archivo(nombre,CONFIG['URLS'][nombre],referencia); particion='train' if 'Training' in nombre else 'test'; listados=extraer_zip(ruta,particion)
        interno='ISIC_2019_Training_Input' if particion=='train' else 'ISIC_2019_Test_Input'; reales=len(list((BASE_DIR/'data_cruda'/particion/interno).glob('*.jpg'))); assert reales==listados, f'JPG extraídos != ZIP en {particion}'
        if CONFIG['FULL_DATASET']: assert reales == (25331 if particion=='train' else 8238)
        inventario.append({'nombre':nombre,'url':CONFIG['URLS'][nombre],'tamano_esperado':observado,'tamano_referencia':referencia,'tamano_observado':observado,'estado':estado,'jpg_zip':listados,'jpg_extraidos':reales})
        if CONFIG['DELETE_ZIPS_AFTER_EXTRACT']: ruta.unlink() # libera ~12.7 GB en Colab
    (BASE_DIR/'inventario_descarga.json').write_text(json.dumps(inventario,ensure_ascii=False,indent=2),encoding='utf8'); return inventario

if __name__ == '__main__': main()
